# 11 · Datum und Zeit

**Lernziele**

- Du arbeitest mit Kalenderdaten (`date`) und Zeitpunkten (`datetime`).
- Du rechnest mit Zeitspannen (`timedelta`).
- Du gibst Daten lesbar aus und liest sie aus Text ein.
- Du kennst den Unterschied zwischen naiven und zeitzonenbewussten Zeitpunkten und rechnest mit `zoneinfo` zwischen Zeitzonen um.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 10.

Rechnungen werden fällig, Kurse beginnen, Logdateien haben Zeitstempel: Mit Datum und Uhrzeit arbeitet fast jedes Programm. Python bringt dafür das Modul `datetime` mit. Die wichtigsten Klassen:

| Klasse | steht für | Beispiel |
| --- | --- | --- |
| `date` | Kalenderdatum | 16.11.2026 |
| `time` | Uhrzeit | 09:30 |
| `datetime` | Datum mit Uhrzeit | 16.11.2026, 09:30 |
| `timedelta` | Zeitspanne | 3 Tage, 2 Stunden |

Modul und Klasse heißen beide `datetime`. Damit nichts durcheinandergeht, importieren wir die Klassen direkt: `from datetime import date, datetime`.

## Ein Kalenderdatum: `date`

In [ ]:
from datetime import date, datetime, time, timedelta

release = date(2026, 10, 9)  # Jahr, Monat, Tag
print(release)
print(release.year, release.month, release.day)
print(release.weekday())  # 0 = Montag … 6 = Sonntag
print(date.today())

`date` prüft schon beim Anlegen, ob es das Datum gibt, Schaltjahre eingeschlossen:

In [ ]:
date(2026, 2, 30)

Daten lassen sich wie Zahlen vergleichen und sortieren. Früher ist kleiner:

In [ ]:
holidays = [date(2026, 12, 25), date(2026, 10, 3), date(2026, 5, 1)]
print(sorted(holidays))
print(date(2026, 10, 3) < date(2026, 12, 25))

**Kurzübung 1 (Feiertag am Wochenende?):** Prüfe für jeden Feiertag in `holidays`, ob er 2026 auf ein Wochenende fällt. Gib die Feiertage zeitlich sortiert aus, etwa `2026-10-03 Wochenende`.

In [ ]:
for holiday in sorted(holidays):
    if holiday.weekday() >= 5:  # 5 = Samstag, 6 = Sonntag
        print(holiday, "Wochenende")
    else:
        print(holiday, "Werktag")
# Nur der Tag der Deutschen Einheit fällt 2026 auf einen Samstag.

## Rechnen mit Zeitspannen: `timedelta`

Ziehst du zwei Daten voneinander ab, erhältst du eine **Zeitspanne** vom Typ `timedelta`. `.days` liefert die Anzahl der Tage als Zahl:

In [ ]:
order_date = date(2026, 10, 30)
course_start = date(2026, 11, 16)

waiting = course_start - order_date
print(waiting, type(waiting))
print(f"Noch {waiting.days} Tage bis zum Kurs")

Umgekehrt ergibt Datum plus Zeitspanne ein neues Datum. `timedelta` versteht `weeks`, `days`, `hours`, `minutes` und `seconds`:

In [ ]:
due_date = order_date + timedelta(days=30)
print(f"Zahlbar bis {due_date}")
print(order_date - timedelta(weeks=2))

> **Typischer Fehler:** `timedelta` kennt keine Monate und Jahre, weil sie unterschiedlich lang sind. „Ein Jahr später“ legst du mit `replace()` selbst fest. Vorsicht bei Tagen, die es im Zielmonat nicht gibt, etwa dem 31. oder dem 29. Februar: Dann gibt es einen `ValueError`.

In [ ]:
print(order_date.replace(year=2027))
print(order_date.replace(month=11))  # den 30.11. gibt es, den 31.11. nicht

**Kurzübung 2 (Rückgabefrist):** Ein Paket kommt am 30.10.2026 an, du kannst es innerhalb von 14 Tagen zurückschicken. Berechne den letzten Tag der Frist. Welche Ausgabe erzeugt danach `date(2026, 1, 31).replace(month=2)`? Begründe deine Antwort, bevor du es ausprobierst.

In [ ]:
delivery = date(2026, 10, 30)
print(delivery + timedelta(days=14))  # 2026-11-13

In [ ]:
date(2026, 1, 31).replace(month=2)  # ValueError: Einen 31. Februar gibt es nicht

## Datum und Uhrzeit: `datetime`

`datetime` verbindet ein Datum mit einer Uhrzeit. Es kann alles, was `date` kann, dazu Stunden, Minuten und Sekunden:

In [ ]:
kickoff = datetime(2026, 11, 16, 9, 30)  # 16.11.2026, 9:30 Uhr
print(kickoff)
print(kickoff.date(), kickoff.time())
print(kickoff + timedelta(hours=1, minutes=45))

Wie lange dauert ein Kurstag? Die Differenz zweier Zeitpunkte ist wieder ein `timedelta`. `total_seconds()` liefert die gesamte Dauer in Sekunden:

In [ ]:
start = datetime(2026, 11, 16, 9, 0)
end = datetime(2026, 11, 16, 16, 30)
duration = end - start
print(duration)
print(f"{duration.total_seconds() / 3600} Stunden")

Hast du Datum und Uhrzeit getrennt, fügt `datetime.combine()` sie zusammen. `datetime.now()` liefert den aktuellen Zeitpunkt:

In [ ]:
print(datetime.combine(date(2026, 12, 24), time(18, 0)))
print(datetime.now())

**Kurzübung 3 (Fahrzeit im Nachtzug):** Der Nachtzug fährt am 02.11.2026 um 22:40 Uhr ab und kommt am nächsten Morgen um 06:15 Uhr an. Wie lange dauert die Fahrt? Gib die Dauer als `timedelta` und in Minuten aus.

In [ ]:
departure = datetime(2026, 11, 2, 22, 40)
arrival = datetime(2026, 11, 3, 6, 15)
trip = arrival - departure
print(trip)  # 7:35:00
print(f"{trip.total_seconds() / 60:.0f} Minuten")  # 455 Minuten
# Mit vollständigen Zeitpunkten klappt die Rechnung über Mitternacht von selbst.

## Ausgeben: ISO 8601 und eigene Formate

Für Dateien, Datenbanken und den Austausch zwischen Programmen gibt es ein internationales Format: **ISO 8601**. Ein Datum schreibt es als Jahr-Monat-Tag, etwa `2026-11-16`. Kommt eine Uhrzeit dazu, steht ein `T` dazwischen: `2026-11-16T09:30:00`. Das Format ist eindeutig und lässt sich sogar als Text richtig sortieren. `print()` und `isoformat()` nutzen es (`print()` mit Leerzeichen statt `T`).

Für Menschen formatierst du selbst, mit `strftime()` (*string format time*) oder direkt im f-String nach einem Doppelpunkt:

| Code | Bedeutung | Beispiel |
| --- | --- | --- |
| `%d` | Tag, zweistellig | `16` |
| `%m` | Monat, zweistellig | `11` |
| `%Y` | Jahr, vierstellig | `2026` |
| `%H` | Stunde (24 h) | `09` |
| `%M` | Minute | `30` |
| `%S` | Sekunde | `00` |
| `%A` | Wochentag | `Monday` |
| `%B` | Monatsname | `November` |

In [ ]:
print(kickoff.isoformat())
print(kickoff.strftime("%d.%m.%Y %H:%M"))
print(f"{kickoff:%d.%m.%Y um %H:%M Uhr}")

`%A` und `%B` liefern die Namen auf Englisch, auch auf einem deutschen Rechner: Python stellt die Sprache für Datumsangaben nur um, wenn ein Programm das ausdrücklich verlangt. Für verlässlich deutsche Namen nimmst du eine eigene Liste und `weekday()` als Index:

In [ ]:
WEEKDAYS = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]

print(f"{WEEKDAYS[kickoff.weekday()]}, {kickoff:%d.%m.%Y}")

**Kurzübung 4 (Dateiname mit Zeitstempel):** Ein Bericht wird am 02.11.2026 um 14:05 Uhr erstellt. Erzeuge daraus den Dateinamen `bericht_2026-11-02_1405.pdf`. Warum sortiert ein Ordner solche Namen zeitlich richtig, Namen wie `bericht_02.11.2026.pdf` aber nicht?

In [ ]:
created = datetime(2026, 11, 2, 14, 5)
print(f"bericht_{created:%Y-%m-%d_%H%M}.pdf")

# Text wird Zeichen für Zeichen sortiert. Bei JJJJ-MM-TT steht das Wichtigste vorn,
# bei TT.MM.JJJJ der Tag: Der 02.11.2026 landet vor dem 15.10.2026.
print(sorted(["bericht_02.11.2026.pdf", "bericht_15.10.2026.pdf", "bericht_01.12.2025.pdf"]))
print(sorted(["bericht_2026-11-02.pdf", "bericht_2026-10-15.pdf", "bericht_2025-12-01.pdf"]))

## Einlesen: aus Text wird ein Datum

Daten kommen oft als Text an, aus einer Datei oder von einer Eingabe. Für ISO-Text gibt es `fromisoformat()`, für alle anderen Formate `strptime()` (*string parse time*) mit denselben Codes wie oben:

In [ ]:
print(date.fromisoformat("2026-12-24"))
print(datetime.fromisoformat("2026-12-24T18:00"))
print(time.fromisoformat("18:00"))  # nur eine Uhrzeit
print(datetime.strptime("24.12.2026 18:00", "%d.%m.%Y %H:%M"))
print(date.strptime("24.12.2026", "%d.%m.%Y"))  # seit Python 3.14

Passt der Text nicht zum Format, gibt es einen `ValueError`. Bei Benutzereingaben fängst du ihn mit `try` / `except` ab (Notebook 09):

In [ ]:
datetime.strptime("2026-12-24", "%d.%m.%Y")

**Kurzübung 5 (Datum aus einem deutschen Export):** Ein Export aus der Buchhaltung liefert Rechnungsdaten als Text wie `"31.10.2026"`. Wandle jedes Datum in `invoice_dates` in ISO-Text wie `2026-10-31` um.

- **Untersuchen:** Was passiert mit `"31.11.2026"`, und warum?

In [ ]:
invoice_dates = ["31.10.2026", "02.11.2026", "15.01.2027"]

In [ ]:
for text in invoice_dates:
    print(date.strptime(text, "%d.%m.%Y").isoformat())

# Untersuchen: "31.11.2026" passt zum Format, einen 31. November gibt es aber nicht.
try:
    date.strptime("31.11.2026", "%d.%m.%Y")
except ValueError as error:
    print("ungültig:", error)

## Zeitzonen

Wann ist „9:30 Uhr“? In Berlin, London und New York sind das drei verschiedene Zeitpunkte. Unsere `datetime`-Objekte hatten bisher keine Zeitzone; sie heißen **naiv**. Python weiß bei ihnen nicht, wo auf der Welt die Uhrzeit gilt.

Ein Zeitpunkt mit Zeitzone heißt **zeitzonenbewusst** (*aware*). Die Zeitzonen liefert das Modul `zoneinfo` aus der Standardbibliothek, mit Namen wie `"Europe/Berlin"` oder `"America/New_York"`. Es kennt auch Sommer- und Winterzeit. `astimezone()` rechnet einen Zeitpunkt in eine andere Zone um; der Zeitpunkt bleibt derselbe, nur die Uhrzeit ändert sich.

> Unter Windows braucht `zoneinfo` das Paket `tzdata`. In diesem Projekt ist es schon installiert.

In [ ]:
from datetime import UTC
from zoneinfo import ZoneInfo

berlin = ZoneInfo("Europe/Berlin")
new_york = ZoneInfo("America/New_York")

kickoff = datetime(2026, 11, 16, 9, 30, tzinfo=berlin)
print(kickoff)
print(kickoff.astimezone(new_york))
print(kickoff.astimezone(UTC))

`+01:00` ist der Abstand zur Weltzeit **UTC**. Im Sommer gilt in Berlin die Sommerzeit, dann sind es zwei Stunden. Den passenden Abstand für jedes Datum kennt `zoneinfo`:

In [ ]:
summer = datetime(2026, 7, 1, 9, 30, tzinfo=berlin)
print(kickoff.utcoffset())  # Winterzeit
print(summer.utcoffset())   # Sommerzeit

Den aktuellen Zeitpunkt holst du zeitzonenbewusst mit `datetime.now(UTC)` oder `datetime.now(berlin)`:

In [ ]:
now_utc = datetime.now(UTC)
print(now_utc)
print(now_utc.astimezone(berlin))

> **Typischer Fehler:** Naive und zeitzonenbewusste Zeitpunkte lassen sich nicht vergleichen und nicht voneinander abziehen. Python weigert sich zu raten:

In [ ]:
datetime.now() < datetime.now(UTC)

**Faustregeln für Zeitzonen:**

- Zeitpunkte, die du speicherst oder mit anderen Systemen austauschst, immer mit Zeitzone, am besten in UTC.
- Erst für die Anzeige in die Zeitzone der Benutzer umrechnen.
- Dauern über eine Zeitumstellung rechnest du über UTC: erst beide Zeitpunkte mit `astimezone(UTC)` umrechnen, dann abziehen. Liegen beide in derselben Zeitzone, zieht Python sonst nur die Uhrzeiten ab, und das Ergebnis liegt um die Stunde der Umstellung daneben.
- `datetime.utcnow()` findest du in älterem Code. Es ist veraltet und liefert trotz des Namens einen naiven Zeitpunkt. Nimm `datetime.now(UTC)`.

**Kurzübung 6 (Ein Termin in drei Zeitzonen):** Ein Termin ist am 16.11.2026 um 16:00 Uhr in Berlin. Wie spät ist es dann in New York (`"America/New_York"`), Tokio (`"Asia/Tokyo"`) und Sydney (`"Australia/Sydney"`)? Achte auf das Datum.

In [ ]:
meeting = datetime(2026, 11, 16, 16, 0, tzinfo=ZoneInfo("Europe/Berlin"))

for zone in ["America/New_York", "Asia/Tokyo", "Australia/Sydney"]:
    local = meeting.astimezone(ZoneInfo(zone))
    print(f"{zone:<20} {local:%d.%m.%Y %H:%M}")

## Ein Kalender auf einen Blick

Für einen schnellen Überblick hat die Standardbibliothek das Modul `calendar`. Die Namen sind englisch:

In [ ]:
import calendar

print(calendar.month(2026, 11))

**Kurzübung 7 (Februar im Schaltjahr):** Gib den Kalender für Februar 2028 aus. Auf welchen Wochentag fällt der 29.02.2028? Lies es im Kalender ab und prüfe es mit `weekday()` und der Liste `WEEKDAYS`.

In [ ]:
print(calendar.month(2028, 2))  # 2028 ist ein Schaltjahr: Der Februar hat 29 Tage
print(WEEKDAYS[date(2028, 2, 29).weekday()])  # Dienstag (Spalte "Tu")

## Übungen

Damit alle dieselbe Ausgabe sehen, gilt in den Übungen `date(2026, 11, 2)` als „heute“.

**Übung 1 (Alter am Stichtag, leicht):** Schreibe `age_on(birthday, reference)`, die das Alter in ganzen Jahren an einem Stichtag liefert. Achtung: Wer im Dezember Geburtstag hat, ist im November noch nicht ein Jahr älter geworden. Erwartet: `age_on(date(1990, 12, 24), date(2026, 11, 2))` ergibt 35, `age_on(date(1990, 3, 1), date(2026, 11, 2))` ergibt 36.

Tipp: Ziehe die Jahre voneinander ab. Vergleiche dann erst den Monat und bei gleichem Monat den Tag: Ist der Geburtstag im Jahr des Stichtags noch nicht erreicht, ziehe 1 ab.

- **Untersuchen:** Wie alt ist jemand, der am 29.02.2000 geboren ist, am 28.02.2026 und am 01.03.2026?

In [ ]:
def age_on(birthday: date, reference: date) -> int:
    years = reference.year - birthday.year
    if reference.month < birthday.month or (
        reference.month == birthday.month and reference.day < birthday.day
    ):
        years -= 1  # Geburtstag ist in diesem Jahr noch nicht gewesen
    return years


print(age_on(date(1990, 12, 24), date(2026, 11, 2)))  # 35
print(age_on(date(1990, 3, 1), date(2026, 11, 2)))  # 36

# Kürzer: Tupel werden Element für Element verglichen, erst der Monat, dann der Tag.
# (reference.month, reference.day) < (birthday.month, birthday.day)

# Untersuchen: Am 28.02. ist der Tag 28 < 29, also noch 25; am 01.03.2026 dann 26.
# Ohne 29. Februar im Jahr 2026 zählt der Geburtstag hier erst ab dem 1. März.
print(age_on(date(2000, 2, 29), date(2026, 2, 28)), age_on(date(2000, 2, 29), date(2026, 3, 1)))

**Übung 2 (Tage zählen und Wochentag finden, leicht):**

1. Wie viele Tage sind es von heute (`today = date(2026, 11, 2)`) bis zum Kursende am 06.11.2026 und bis Heiligabend (24.12.2026)? Ziehe die Daten voneinander ab und nimm `.days`.
2. Auf welchen Wochentag fällt dein Geburtstag im Jahr 2027? Gib den Wochentag auf Deutsch mit der Liste `WEEKDAYS` aus, etwa `Der 14.03.2027 ist ein Sonntag`. Setze statt des Beispiels deinen eigenen Geburtstag ein.

In [ ]:
today = date(2026, 11, 2)
course_end = date(2026, 11, 6)
christmas_eve = date(2026, 12, 24)

print(f"Noch {(course_end - today).days} Tage bis zum Kursende")  # 4
print(f"Noch {(christmas_eve - today).days} Tage bis Heiligabend")  # 52

birthday = date(2027, 3, 14)
print(f"Der {birthday:%d.%m.%Y} ist ein {WEEKDAYS[birthday.weekday()]}")

**Übung 3 (Überfällige Rechnungen, leicht):** Jede Rechnung in `invoices` ist 14 Tage nach dem Rechnungsdatum fällig. Gib für heute (`date(2026, 11, 2)`) jede überfällige Rechnung mit der Zahl der Tage seit der Fälligkeit aus, etwa `R-1000: seit 34 Tagen überfällig`.

- **Untersuchen:** Ist die Rechnung, die genau heute fällig wird, schon überfällig? Welcher Vergleich, `>` oder `>=`, drückt das richtig aus?

In [ ]:
invoices = {
    "R-1000": date(2026, 9, 15),
    "R-1001": date(2026, 10, 5),
    "R-1002": date(2026, 10, 19),
    "R-1003": date(2026, 10, 29),
}

In [ ]:
today = date(2026, 11, 2)
for number, invoice_date in invoices.items():
    due = invoice_date + timedelta(days=14)
    if today > due:
        print(f"{number}: seit {(today - due).days} Tagen überfällig")
# R-1000: seit 34 Tagen überfällig
# R-1001: seit 14 Tagen überfällig

# Untersuchen: R-1002 ist genau heute fällig. Bezahlt werden darf bis einschließlich
# Fälligkeitstag, überfällig ist sie also erst morgen: today > due, nicht >=.

**Übung 4 (Termine sortieren, mittel):** Lies die Termine aus `appointments` ein, sortiere sie zeitlich und gib sie mit deutschem Wochentag aus, etwa `Montag, 16.11.2026 09:30`. Als Text sortiert stünde der 05.01.2027 an zweiter Stelle; erst als `datetime` stimmt die Reihenfolge.

Es gibt zwei Wege: erst alle Texte in `datetime` umwandeln und diese Liste sortieren, oder die Texte mit `sorted(..., key=...)` sortieren. Welchen nimmst du, und warum?

In [ ]:
appointments = ["24.12.2026 18:00", "16.11.2026 09:30", "05.01.2027 10:00", "03.10.2026 14:00"]

In [ ]:
print(sorted(appointments))  # als Text: 05.01.2027 an zweiter Stelle

# Weg 1: erst umwandeln, dann sortieren
parsed = [datetime.strptime(text, "%d.%m.%Y %H:%M") for text in appointments]
for appointment in sorted(parsed):
    print(f"{WEEKDAYS[appointment.weekday()]}, {appointment:%d.%m.%Y %H:%M}")

# Weg 2: die Texte sortieren, verglichen wird aber das umgewandelte Datum
for text in sorted(appointments, key=lambda text: datetime.strptime(text, "%d.%m.%Y %H:%M")):
    print(text)

# Weg 1 ist hier besser: Für die Ausgabe mit Wochentag brauchst du die datetime-Werte
# ohnehin. Weg 2 passt, wenn die Texte unverändert bleiben sollen.

**Übung 5 (Nachtdienste über die Zeitumstellung, mittel):** In `shifts` stehen drei Nachtdienste in Berlin, je Beginn und Ende. Berechne für jeden Dienst die Dauer einmal direkt (Ende minus Beginn) und einmal über UTC, und gib aus, wie viele Stunden insgesamt bezahlt werden. Bei welchen Diensten weichen die beiden Rechnungen ab, und warum? Erwartet: 24 Stunden bezahlt.

In [ ]:
berlin = ZoneInfo("Europe/Berlin")
shifts = [
    (datetime(2026, 3, 27, 22, 0, tzinfo=berlin), datetime(2026, 3, 28, 6, 0, tzinfo=berlin)),
    (datetime(2026, 3, 28, 22, 0, tzinfo=berlin), datetime(2026, 3, 29, 6, 0, tzinfo=berlin)),
    (datetime(2026, 10, 24, 22, 0, tzinfo=berlin), datetime(2026, 10, 25, 6, 0, tzinfo=berlin)),
]

In [ ]:
paid = timedelta()
for shift_start, shift_end in shifts:
    wall_clock = shift_end - shift_start
    actual = shift_end.astimezone(UTC) - shift_start.astimezone(UTC)
    paid += actual
    print(f"{shift_start:%d.%m.%Y}: direkt {wall_clock}, über UTC {actual}")
print(f"Bezahlt: {paid.total_seconds() / 3600:.0f} Stunden")  # 24 Stunden

# Direkt gerechnet zeigt jeder Dienst 8:00:00, denn Python zieht in derselben
# Zeitzone die Uhrzeiten nach Wanduhr ab. Über UTC zählen die echten Stunden:
# In der Nacht zum 29.03.2026 springt die Uhr von 2:00 auf 3:00 Uhr (7 Stunden),
# in der Nacht zum 25.10.2026 von 3:00 zurück auf 2:00 Uhr (9 Stunden).
# Regel: Dauer immer in UTC rechnen.

**Übung 6 (Zahlungsfrist in Werktagen: KI-Code prüfen, mittel):** Eine Rechnung ist eine Anzahl Werktage nach dem Rechnungsdatum fällig. Eine Chat-KI hat dafür `add_workdays(start, workdays, holidays)` geschrieben. Die Funktion soll das Datum liefern, das `workdays` Werktage nach `start` liegt; Samstage, Sonntage und Tage aus `holidays` zählen nicht, der Starttag auch nicht.

Lies den Code und sage vorher, welche der drei Prüfungen `True` und welche `False` ausgibt. Führe die Zelle dann aus, finde die Fehler und behebe sie, bis alle drei Prüfungen `True` ausgeben.

- **Untersuchen:** Behebe zuerst nur einen Fehler. Warum gibt die dritte Prüfung dann immer noch `False` aus?

In [ ]:
from datetime import date, timedelta


# Antwort einer Chat-KI
def add_workdays(start, workdays, holidays):
    """Liefert das Datum, das workdays Werktage nach start liegt."""
    day = start
    while workdays > 0:
        day += timedelta(days=1)
        if day.weekday() <= 5 and day.isoformat() not in holidays:
            workdays -= 1
    return day


holidays = {date(2026, 12, 24), date(2026, 12, 25), date(2026, 12, 31), date(2027, 1, 1)}
print(add_workdays(date(2026, 11, 23), 3, holidays) == date(2026, 11, 26))  # Montag + 3 = Donnerstag
print(add_workdays(date(2026, 11, 27), 1, holidays) == date(2026, 11, 30))  # Freitag + 1 = Montag
print(add_workdays(date(2026, 12, 18), 10, holidays) == date(2027, 1, 7))  # über Weihnachten und Neujahr

In [ ]:
# Vorhersage: True, False, False.
# Fehler 1: weekday() <= 5 lässt den Samstag (5) als Werktag durch. Richtig: < 5.
# Fehler 2: day.isoformat() ist Text, holidays enthält aber date-Werte. Text ist nie
# gleich einem date, also zählt jeder Feiertag als Werktag. Richtig: day not in holidays.
# Untersuchen: Mit nur Fehler 1 behoben landet die dritte Prüfung beim 01.01.2027,
# weil die vier Feiertage weiter mitgezählt werden.
def add_workdays(start, workdays, holidays):
    """Liefert das Datum, das workdays Werktage nach start liegt."""
    day = start
    while workdays > 0:
        day += timedelta(days=1)
        if day.weekday() < 5 and day not in holidays:
            workdays -= 1
    return day


print(add_workdays(date(2026, 11, 23), 3, holidays) == date(2026, 11, 26))  # True
print(add_workdays(date(2026, 11, 27), 1, holidays) == date(2026, 11, 30))  # True
print(add_workdays(date(2026, 12, 18), 10, holidays) == date(2027, 1, 7))  # True

**Übung 7 (Arbeitszeit und Überstunden, schwer):** Deine Stempelzeiten stehen in `week` (Datum -> Liste von Kommen/Gehen-Paaren). Berechne die Arbeitszeit je Tag und die Abweichung von 8 Stunden. Erwartet:

```text
Mo 16.11.: 8:26 h (+0:26)
Di 17.11.: 6:30 h (-1:30)
```

Welche Ausgabe erzeugt `print(timedelta(minutes=-30))`, und warum brauchst du eine eigene Formatierung?

Tipp: `time.fromisoformat("08:12")` liest eine Uhrzeit, `datetime.combine(day, ...)` macht daraus einen Zeitpunkt. Rechne für die Ausgabe in ganze Minuten um und teile mit `divmod(minutes, 60)` in Stunden und Minuten.

In [ ]:
week = {
    date(2026, 11, 16): [("08:12", "12:03"), ("12:45", "17:20")],
    date(2026, 11, 17): [("09:00", "12:30"), ("13:00", "16:00")],
}

In [ ]:
SHORT_WEEKDAYS = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]


def as_hours(duration: timedelta, with_sign: bool = False) -> str:
    """Formatiert eine Dauer als Stunden:Minuten, etwa 8:26 oder mit Vorzeichen +0:26."""
    minutes = int(duration.total_seconds() // 60)
    hours, rest = divmod(abs(minutes), 60)
    if minutes < 0:
        sign = "-"
    elif with_sign:
        sign = "+"
    else:
        sign = ""
    return f"{sign}{hours}:{rest:02d}"


for day, blocks in week.items():
    worked = timedelta()
    for come, leave in blocks:
        arrival = datetime.combine(day, time.fromisoformat(come))
        departure = datetime.combine(day, time.fromisoformat(leave))
        worked += departure - arrival
    overtime = worked - timedelta(hours=8)
    print(f"{SHORT_WEEKDAYS[day.weekday()]} {day:%d.%m.}: {as_hours(worked)} h ({as_hours(overtime, with_sign=True)})")

print(timedelta(minutes=-30))  # -1 day, 23:30:00
# timedelta speichert negative Dauern als "minus einen Tag plus Rest".
# Das ist korrekt, aber unlesbar; deshalb die eigene Formatierung as_hours.

**Übung 8 (Fotos nach Monat einsortieren, schwer, für Schnelle):** Lies aus jedem Fotonamen der Form `IMG_20261030_081502.jpg` das Aufnahmedatum mit `strptime` und verschiebe das Foto in einen Unterordner wie `2026-10`. Fotos ohne lesbares Datum bekommen den Monat ihrer letzten Änderung (`stat().st_mtime`). Es gelten die Regeln wie beim Download-Ordner in Notebook 10: erst ein Trockenlauf, nichts überschreiben (ein Foto, das es am Ziel schon gibt, wird gemeldet), und die Endung zählt in jeder Schreibweise (`.jpg` wie `.JPG`). Andere Dateien bleiben liegen.

Tipp: `path.suffix.lower()` liefert die Endung kleingeschrieben. `path.name.removesuffix(path.suffix)` entfernt die Endung, wie `removeprefix` aus Notebook 05, nur am Ende. `datetime.fromtimestamp(path.stat().st_mtime)` macht aus der Änderungszeit einen Zeitpunkt.

Die erste Zelle legt einen Beispielordner mit Fotos in einem temporären Ordner an. Du kannst sie beliebig oft ausführen; jedes Mal entsteht ein frischer Ordner.

In [ ]:
import os
import tempfile
from pathlib import Path

# Beispielordner in einem temporären Verzeichnis: keine Spuren im Projekt
photo_dir = tempfile.TemporaryDirectory()
photos = Path(photo_dir.name)

names = [
    "IMG_20261030_081502.jpg",
    "IMG_20261101_174233.jpg",
    "IMG_20261102_093000.JPG",  # Endung großgeschrieben
    "IMG_20260915_120000.jpg",
    "IMG_20261340_101010.jpg",  # Monat 13: kein gültiges Datum
    "urlaub_strand.jpg",
    "liesmich.txt",  # kein Foto: bleibt liegen
]
changed = datetime(2026, 8, 15, 12, 0).timestamp()  # feste letzte Änderung
for name in names:
    path = photos / name
    path.write_bytes(b"")
    os.utime(path, (changed, changed))

# Dieses Foto liegt schon im Zielordner und darf nicht überschrieben werden
(photos / "2026-10").mkdir()
(photos / "2026-10" / "IMG_20261030_081502.jpg").write_bytes(b"schon einsortiert")

print(sorted(path.name for path in photos.iterdir()))

In [ ]:
def sort_photos(photos: Path, dry_run: bool) -> None:
    for path in sorted(photos.iterdir()):
        if not path.is_file() or path.suffix.lower() != ".jpg":
            continue  # Ordner und andere Dateien bleiben liegen
        try:
            taken = datetime.strptime(path.name.removesuffix(path.suffix), "IMG_%Y%m%d_%H%M%S")
        except ValueError:  # kein lesbares Datum im Namen
            taken = datetime.fromtimestamp(path.stat().st_mtime)
        target = photos / f"{taken:%Y-%m}" / path.name
        if target.exists():
            print(f"übersprungen, existiert schon: {target.parent.name}/{target.name}")
        elif dry_run:
            print(f"würde verschieben: {path.name} -> {target.parent.name}/")
        else:
            target.parent.mkdir(exist_ok=True)
            path.rename(target)


sort_photos(photos, dry_run=True)

# Erst wenn der Trockenlauf stimmt, wirklich verschieben:
sort_photos(photos, dry_run=False)
for folder in sorted(photos.iterdir()):
    if folder.is_dir():
        for path in sorted(folder.iterdir()):
            print(f"{folder.name}/{path.name}")

## Zusammenfassung

- `date` für Kalenderdaten, `datetime` für Zeitpunkte, `time` für Uhrzeiten.
- Die Differenz zweier Daten ist ein `timedelta`; Datum ± `timedelta` ergibt ein Datum. Monate und Jahre mit `replace()`.
- Ausgabe: `isoformat()` für Maschinen, `strftime()` oder `f"{d:%d.%m.%Y}"` für Menschen. `%A` und `%B` sind englisch.
- Einlesen: `fromisoformat()` für ISO-Text, `strptime()` für alle anderen Formate.
- Zeitzonen mit `ZoneInfo("Europe/Berlin")`. In UTC speichern, lokal anzeigen. `datetime.now(UTC)` statt `utcnow()`.
- Dauern über eine Zeitumstellung über UTC rechnen.
- Naive und zeitzonenbewusste Zeitpunkte nie mischen.